<a href="https://colab.research.google.com/github/SEC-API-io/sec-api-cookbook/blob/main/notebooks/material-contracts/download-material-contracts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download Material Contracts (Exhibit 10) from SEC 10-K Filings

This example demonstrates how to locate material contracts disclosed in Exhibit 10 of SEC filings—such as Forms 10-K, 10-Q, or S-1—and download them in their original HTML format and as PDFs to local disk.

For this example, the focus is on Exhibit 10 contracts disclosed in Form 10-K filings from 2020 to 2023. The steps are as follows:

1. Use the Query API to find URLs for Exhibit 10 files within 10-K filings.
2. Download the material contracts as both HTML and PDF files using the Filing Download and PDF Generator APIs.

In [ ]:
pip install sec-api

In [ ]:
SEC_API_KEY = "YOUR_API_KEY"

### Find and Aggregate URLs of Exhibit 10 Files from 10-K Filings

The Query API is used to locate and compile URLs for material contracts disclosed in Exhibit 10 from 10-K filings submitted between 2020 and 2023. The following search query filters the desired filings:

```
formType:"10-K" AND documentFormatFiles.type:"EX-10" AND filedAt:[2020-01-01 TO 2023-12-31]
```

This search retrieves metadata for all Form 10-K filings containing `EX-10` (Exhibit 10) documents filed within the specified date range (January 2020 to December 2023). The `documentFormatFiles` array within each filing’s metadata includes detailed information about each attached document, such as its URL (`documentUrl`), type, description, and size. An example structure of the `documentFormatFiles` array is shown below:

```json
"documentFormatFiles": [
  {
    "sequence": "3",
    "size": "50752",
    "documentUrl": "https://www.sec.gov/Archives/edgar/data/72331/000007233123000242/ex10-unordsonxformofstocko.htm",
    "description": "EX-10.U",
    "type": "EX-10.U"
  }
  // ... additional documents
],
// ... other filing metadata
```
<br/>

**Examples of Exhibit 10 Files**

Exhibit 10 material contracts encompass a variety of agreement types, including but not limited to:

- [Compensation plans for non-employee directors](https://www.sec.gov/Archives/edgar/data/29644/000002964424000111/exhibit10-aadci20240731.htm)
- [Stock award notices](https://www.sec.gov/Archives/edgar/data/1067294/000155837024013061/tmb-20240802xex10dj.htm)
- [Board resolutions (e.g., issuance of convertible notes)](https://www.sec.gov/Archives/edgar/data/1788841/000107997324000555/ex10x21.htm)
- [Investment management trust agreements](https://www.sec.gov/Archives/edgar/data/1849548/000200858924000010/f10k2023ex10-10.htm)
- [Credit card program agreements](https://www.sec.gov/Archives/edgar/data/28917/000002891724000011/dds-20240203xex10dk.htm)
- [License agreements](https://www.sec.gov/Archives/edgar/data/1512762/000155837024003436/chrs-20231231xex10d40.htm)
- [Credit agreements](https://www.sec.gov/Archives/edgar/data/1474903/000147490324000026/ex10242023.htm)
- [Note purchase agreements](https://www.sec.gov/Archives/edgar/data/18672/000108981924000006/cnl-20231231x10kxex10h.htm)
- [Share exchange agreements](https://www.sec.gov/Archives/edgar/data/1883835/000152013824000304/esgh-20240918_s1aex10z4.htm)

Although the Query API locates filings containing Exhibit 10, it cannot filter specific types of material contracts within Exhibit 10. To identify specific contract types, additional filtering can be performed client-side by downloading the HTML content and searching for specific keywords or phrases, such as "license agreement," within the exhibit text.

In [ ]:
import pandas as pd
from sec_api import QueryApi

queryApi = QueryApi(api_key=SEC_API_KEY)

In [ ]:
query = 'formType:"10-K" AND documentFormatFiles.type:"EX-10" AND filedAt:[2020-01-01 TO 2023-12-31]'

search_params = {
    "query": query,
    "from": 0,
    "size": 50,
    "sort": [{"filedAt": {"order": "desc"}}],
}

response = queryApi.get_filings(search_params)

print(f'Number of 10-K filings with Exhibit 10 (2020-2023) found:')
print(response["total"]["value"])

Number of 10-K filings with Exhibit 10 (2020-2023) found:
923


In [ ]:
import re


def is_exhibit_10(file):
    return bool(re.search(r"EX-10", file["type"]))


def get_exhibit_10_urls():
    exhibits = []
    has_more_filings = True

    query = 'formType:"10-K" AND documentFormatFiles.type:"EX-10" AND filedAt:[2020-01-01 TO 2023-12-31]'

    search_params = {
        "query": query,
        "from": 0,
        "size": 50,
        "sort": [{"filedAt": {"order": "desc"}}],
    }

    while has_more_filings:
        response = queryApi.get_filings(search_params)

        if len(response["filings"]) == 0:
            break

        for filing in response["filings"]:
            for file in filing["documentFormatFiles"]:
                if is_exhibit_10(file):
                    exhibits.append(
                        {
                            "accessionNo": filing["accessionNo"],
                            "filedAt": filing["filedAt"],
                            "companyName": filing["companyName"],
                            "ticker": filing["ticker"],
                            "cik": filing["cik"],
                            "exhibit10Url": file["documentUrl"],
                        }
                    )

        search_params["from"] += 50

    return pd.DataFrame(exhibits)


exhibit_10_files = get_exhibit_10_urls()

print("Exhibit 10 files:")
exhibit_10_files

Exhibit 10 files:


,accessionNo,filedAt,companyName,ticker,cik,exhibit10Url
0,0000072331-23-000242,2023-12-20T17:04:04-05:00,NORDSON CORP,NDSN,72331,https://www.sec.gov/Archives/edgar/data/72331/...
1,0000072331-23-000242,2023-12-20T17:04:04-05:00,NORDSON CORP,NDSN,72331,https://www.sec.gov/Archives/edgar/data/72331/...
2,0000072331-23-000242,2023-12-20T17:04:04-05:00,NORDSON CORP,NDSN,72331,https://www.sec.gov/Archives/edgar/data/72331/...
3,0000072331-23-000242,2023-12-20T17:04:04-05:00,NORDSON CORP,NDSN,72331,https://www.sec.gov/Archives/edgar/data/72331/...
4,0001437749-23-034783,2023-12-18T17:26:13-05:00,HOVNANIAN ENTERPRISES INC,HOV,357294,https://www.sec.gov/Archives/edgar/data/357294...
...,...,...,...,...,...,...
2796,0001133421-20-000006,2020-01-30T06:39:26-05:00,NORTHROP GRUMMAN CORP /DE/,NOC,1133421,https://www.sec.gov/Archives/edgar/data/113342...
2797,0001067701-20-000008,2020-01-29T16:35:59-05:00,"UNITED RENTALS, INC.",URI,1067701,https://www.sec.gov/Archives/edgar/data/104716...
2798,0001067701-20-000008,2020-01-29T16:35:59-05:00,UNITED RENTALS NORTH AMERICA INC,URI,1047166,https://www.sec.gov/Archives/edgar/data/104716...
2799,0001564590-20-002467,2020-01-29T16:32:43-05:00,SYNNEX CORP,SNX,1177394,https://www.sec.gov/Archives/edgar/data/117739...


In [ ]:
exhibit_10_files['exhibit10Url'][:10].to_list()

['https://www.sec.gov/Archives/edgar/data/72331/000007233123000242/ex10-unordsonxformofstocko.htm',
 'https://www.sec.gov/Archives/edgar/data/72331/000007233123000242/ex10-vnordsonxformofrestri.htm',
 'https://www.sec.gov/Archives/edgar/data/72331/000007233123000242/ex10-wnoticeofawardpsufy24.htm',
 'https://www.sec.gov/Archives/edgar/data/72331/000007233123000242/ex10-xrestrictedshareunita.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_606399.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_605281.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_605282.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_605283.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_605284.htm',
 'https://www.sec.gov/Archives/edgar/data/357294/000143774923034783/ex_605285.htm']

### Download Material Contracts as HTML and PDF Files

This final step illustrates how to download both HTML and PDF versions of material contracts disclosed in Exhibit 10. The Filing Download and PDF Generator APIs facilitate the file downloads, while `pandarallel` is used to parallelize the process, enabling concurrent downloads for improved speed.

The downloaded Exhibit 10 files are organized into a structured folder hierarchy as follows:

```
exhibit_10_files/<cik>/<accession_number>/<exhibit_filename>.(htm|pdf)
```

Example folder structure:

```
exhibit_10_files/
    72331/
        000007233123000242/
            ex10-stock-options.htm
            ex10-stock-options.pdf
            ex10-share-unit-awards.htm
            ex10-share-unit-awards.pdf
        ...
    ...
```

In [ ]:
pip install pandarallel ipywidgets

In [ ]:
import os
from pandarallel import pandarallel
from sec_api import RenderApi, PdfGeneratorApi

pandarallel.initialize(nb_workers=5, progress_bar=True)

renderApi = RenderApi(SEC_API_KEY)
pdfGeneratorApi = PdfGeneratorApi(SEC_API_KEY)


def download_exhibit_10_file(row):
    cik = row["cik"]
    accessionNo = row["accessionNo"]
    url = row["exhibit10Url"]

    file_name_html = url.split("/")[-1]
    file_name_pdf = file_name_html.replace(".htm", ".pdf")

    folder = f"exhibit_10_files/{cik}/{accessionNo}/"

    if not os.path.exists(folder):
        os.makedirs(folder)

    exhibit_file_html = renderApi.get_file(url)
    exhibit_file_pdf = pdfGeneratorApi.get_pdf(url)

    # save HTML and PDF file
    with open(folder + file_name_html, "w") as file:
        file.write(exhibit_file_html)
    with open(folder + file_name_pdf, "wb") as file:
        file.write(exhibit_file_pdf)


# download the first 20 Exhibit 10 files
results = exhibit_10_files[:20].parallel_apply(download_exhibit_10_file, axis=1)
# uncomment to download all Exhibit 10 files
# results = exhibit_10_files.parallel_apply(download_exhibit_10_file, axis=1)

INFO: Pandarallel will run on 5 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.
